In [ ]:
import random
import pandas as pd

intents = {
    "calculation": [
        "What is the tax for {} income",
        "Calculate tax for {} salary",
        "How much tax do I pay for {}",
        "Tax amount for {} per year",
        "If I earn {}, what is my tax"
    ],
    "deduction": [
        "What is section 80C",
        "Explain 80C deduction",
        "How to save tax using 80C",
        "Tell me about tax deductions",
        "What deductions are available"
    ],
    "filing": [
        "How to file ITR",
        "Steps to file income tax",
        "How do I submit tax return",
        "Process of tax filing",
        "Where to file ITR"
    ],
    "deadline": [
        "What is the last date to file tax",
        "Tax filing deadline",
        "When should I submit ITR",
        "Due date for tax filing",
        "Last date for income tax return"
    ],
    "general": [
        "What is income tax",
        "Explain tax system",
        "Why do we pay tax",
        "What is taxable income",
        "Tell me about taxes"
    ]
}

dataset = []

for _ in range(10000):
    intent = random.choice(list(intents.keys()))
    query_template = random.choice(intents[intent])

    if "{}" in query_template:
        value = random.choice([
            "5 lakh", "6 lakh", "7 lakh", "8 lakh",
            "10 lakh", "12 lakh", "15 lakh"
        ])
        query = query_template.format(value)
    else:
        query = query_template

    dataset.append([query, intent])

df = pd.DataFrame(dataset, columns=["query", "intent"])

df.to_csv("tax_dataset_10k.csv", index=False)

print("Dataset created with", len(df), "rows")
df.head()

In [ ]:
from google.colab import files
files.download("tax_dataset_10k.csv")

In [ ]:
import pandas as pd
df = pd.read_csv("tax_dataset_10k.csv")
df.head()

In [ ]:
X = df["query"]
y = df["intent"]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_vectorized = vectorizer.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

In [ ]:
accuracy = model.score(X_test, y_test)
print("Model Accuracy:", accuracy)

In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
from google.colab import files

files.download("model.pkl")
files.download("vectorizer.pkl")

In [ ]:
import pickle

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

In [ ]:
def get_intent(user_input):
    text = vectorizer.transform([user_input])
    prediction = model.predict(text)
    return prediction[0]

In [ ]:
import re

def extract_income(text):
    text = str(text).lower().strip()

    numbers = re.findall(r'\d+', text)

    if not numbers:
        return None

    income = int(numbers[0])

    if "lakh" in text:
        income = income * 100000

    return income

In [ ]:
def calculate_tax(income, regime="new"):

    std_deduction = 75000 if regime == "new" else 50000
    taxable_income = max(0, income - std_deduction)

    tax = 0

    if regime == "new":

        slabs = [
            (400000, 0.00),
            (800000, 0.05),
            (1200000, 0.10),
            (1600000, 0.15),
            (2000000, 0.20),
            (2400000, 0.25),
            (float("inf"), 0.30)
        ]

        prev = 0

        for limit, rate in slabs:
            if taxable_income > limit:
                tax += (limit - prev) * rate
                prev = limit
            else:
                tax += (taxable_income - prev) * rate
                break

    else:
        slabs = [
            (250000, 0.00),
            (500000, 0.05),
            (1000000, 0.20),
            (float("inf"), 0.30)
        ]

        prev = 0

        for limit, rate in slabs:
            if taxable_income > limit:
                tax += (limit - prev) * rate
                prev = limit
            else:
                tax += (taxable_income - prev) * rate
                break

    cess = tax * 0.04
    total = tax + cess

    return {
        "taxable_income": taxable_income,
        "tax": round(tax, 2),
        "cess": round(cess, 2),
        "total": round(total, 2)
    }

In [ ]:
import re

qa_map = {
    "what is section 80c": "Section 80C allows deduction up to ₹1.5 lakh on investments like PPF, ELSS, LIC, EPF.",
    "80c deduction": "Section 80C allows deduction up to ₹1.5 lakh on investments like PPF, ELSS, LIC, EPF.",
    "how much 80c limit": "Maximum deduction under Section 80C is ₹1.5 lakh.",

    "what is itr": "ITR is Income Tax Return form used to declare income to the government.",
    "how to file itr": "You can file ITR on the Income Tax e-filing portal using PAN and Aadhaar.",
    "steps to file itr": "Login → Fill form → Verify → Submit.",
    "itr process": "ITR filing involves reporting income, deductions, and paying tax.",

    "itr deadline": "The ITR filing deadline is usually July 31 every year.",
    "last date for itr": "The ITR filing deadline is usually July 31 every year.",
    "due date itr": "The ITR filing deadline is usually July 31 every year.",
    "when to file itr": "ITR should be filed before July 31 each year.",

    "what is income tax": "Income tax is the tax paid to the government based on earnings.",
    "why do we pay tax": "Taxes fund public services like roads, healthcare, and education.",
    "who should pay tax": "Anyone earning above exemption limit must pay tax.",

    "what is gross income": "Gross income is total income before deductions.",
    "what is net salary": "Net salary is income after tax.",
    "what is taxable income": "Taxable income = Income - ₹75,000 deduction.",

    "what is tds": "TDS is tax deducted before salary is credited.",
    "tds meaning": "TDS ensures tax is collected at source.",

    "what is form 16": "Form 16 shows salary details and tax deducted.",
    "why form 16 needed": "Form 16 helps in filing ITR.",

    "what is pan card": "PAN is a unique ID for tax purposes.",
    "why pan needed": "PAN is mandatory for filing ITR.",

    "what is 80d": "Section 80D gives health insurance deduction.",
    "health insurance deduction": "Tax benefits available under Section 80D.",

    "what is rebate 87a": "Section 87A gives rebate for low income.",
    "who gets 87a rebate": "Eligible low-income taxpayers get rebate under 87A.",

    "what is new tax regime": "New regime has lower tax rates but fewer deductions.",
    "what is old tax regime": "Old regime allows deductions like 80C, 80D.",
    "old vs new tax regime": "Old allows deductions, new has lower rates.",

    "what is standard deduction": "Standard deduction is ₹75,000.",

    "documents for itr": "PAN, Aadhaar, Form 16, bank details needed.",
    "what documents needed for itr": "PAN, Aadhaar, Form 16, bank details needed.",

    "miss itr deadline": "Late filing may attract penalty.",
    "penalty for late itr": "Penalty may apply for late ITR filing.",

    "do students pay tax": "Students pay tax only if income exceeds limit.",
    "minimum income for tax": "Income above exemption limit is taxable.",

    "how to reduce tax": "Use deductions like 80C, 80D.",
    "can i save tax": "Yes, through proper tax planning."
}

stopwords = ["what", "is", "how", "to", "the", "for", "a", "of", "do", "we", "are"]

def calculate_tax(income):
    std = 75000
    taxable = max(0, income - std)

    if taxable <= 400000:
        tax = 0
    elif taxable <= 800000:
        tax = (taxable - 400000) * 0.05
    elif taxable <= 1200000:
        tax = 20000 + (taxable - 800000) * 0.10
    elif taxable <= 1600000:
        tax = 60000 + (taxable - 1200000) * 0.15
    else:
        tax = 120000 + (taxable - 1600000) * 0.20

    cess = tax * 0.04
    total = tax + cess

    return {
        "taxable_income": taxable,
        "std_deduction": std,
        "tax": tax,
        "cess": cess,
        "total": total
    }

def extract_income(text):
    text = text.lower()

    match_lakh = re.search(r'(\d+)\s*lakh', text)
    if match_lakh:
        return int(match_lakh.group(1)) * 100000

    match_num = re.search(r'\d+', text)
    if match_num:
        return int(match_num.group())

    return None

def compare_regimes(income):
    old = calculate_tax(income)
    new = calculate_tax(income * 0.9)

    return {
        "old_tax": old["tax"],
        "old_total": old["total"],
        "new_tax": new["tax"],
        "new_total": new["total"],
        "best": "Old Regime" if old["total"] < new["total"] else "New Regime",
        "savings": abs(old["total"] - new["total"])
    }

def generate_insights(result, income):
    slab_line = f"Your taxable income is ₹{result['taxable_income']:,}"
    deduction_line = f"Standard deduction of ₹{result['std_deduction']:,} applied"
    regime_line = "Tax computed using current slab structure"

    suggestions = [
        "Invest in 80C instruments to reduce tax",
        "Consider health insurance for 80D benefits",
        "Compare regimes before filing ITR"
    ]

    return slab_line, deduction_line, regime_line, suggestions

def chatbot_response(user_input):
    user_input = user_input.lower()

    if any(greet in user_input for greet in ["hi", "hello", "hey"]):
        return "👋 Hey! I'm your AI Tax Assistant. Ask me anything about taxes!"

    best_match = None
    best_score = 0

    for question, answer in qa_map.items():
        keywords = [w for w in question.split() if w not in stopwords]
        score = sum(1 for word in keywords if word in user_input)

        if score > best_score:
            best_score = score
            best_match = answer

    if best_score > 0:
        return f"💡 {best_match}"

    income = extract_income(user_input)

    if income and ("tax" in user_input or "income" in user_input):
        result = calculate_tax(income)
        comp = compare_regimes(income)

        slab_line, deduction_line, regime_line, suggestions = generate_insights(result, income)

        return {
            "result": result,
            "comp": comp,
            "slab_line": slab_line,
            "deduction_line": deduction_line,
            "regime_line": regime_line,
            "suggestions": suggestions
        }

    return "🤖 I didn’t understand that. Try asking about tax or 80C."

print("===== Welcome to AI Tax Chatbot =====")

name = input("Enter Name: ")
pan = input("Enter PAN: ")
fy = input("Enter Financial Year (e.g., 2025-26): ")

start = int(fy.split("-")[0])
ay = f"{start+1}-{start+2}"


while True:
    print("\n===== AI Tax Chatbot =====")
    print("1. Calculate Tax")
    print("2. Ask Questions")
    print("3. Exit")

    choice = input("Enter choice (1/2/3): ")

    if choice == "1":
        income_input = input("Enter your income: ")
        income = extract_income(income_input)

        if income:
            result = calculate_tax(income)

            print("\n===== TAX REPORT =====")
            print("Name:", name)
            print("PAN:", pan)
            print("FY:", fy)
            print("AY:", ay)
            print("Taxable Income:", result["taxable_income"])
            print("Tax:", result["tax"])
            print("Cess:", result["cess"])
            print("Total:", result["total"])
        else:
            print("Invalid input")

    elif choice == "2":
        user_input = input("\nAsk your question: ")
        print("Bot:", chatbot_response(user_input))

    elif choice == "3":
        print("Goodbye 👋")
        break

    else:
        print("Invalid choice")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import pandas as pd

years = [f"{y}-{str(y+1)[-2:]}" for y in range(2015, 2025)]

np.random.seed(42)

df = pd.DataFrame({
    "FY": years,

    "Direct_Tax": np.abs(np.linspace(5000, 12000, len(years)) + np.random.randint(-300, 300, len(years))),
    "GST": np.abs(np.linspace(4000, 15000, len(years)) + np.random.randint(-400, 400, len(years))),
    "Customs": np.abs(np.linspace(2000, 7000, len(years)) + np.random.randint(-250, 250, len(years))),
    "Excise": np.abs(np.linspace(3000, 8000, len(years)) + np.random.randint(-300, 300, len(years)))
})

df = df.round(0)

df

In [ ]:
df.info()


In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.sort_values("FY")

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={
    "Direct_Tax_Collection": "Direct_Tax"
})

In [ ]:
df.columns

In [ ]:
if "Customs" not in df.columns:
    df["Customs"] = np.nan

In [ ]:
df.columns[df.columns.duplicated()]
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
num_cols = df.select_dtypes(include="object").columns

for col in ["Direct_Tax", "GST", "Customs", "Excise"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.fillna(df.mean(numeric_only=True))
df = df.drop_duplicates()
df

In [ ]:
df = df.fillna(0)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = df.sort_values("FY")

years = df["FY"].astype(str)

x = np.arange(len(years))
width = 0.2

plt.figure(figsize=(14,6))

plt.bar(x - 1.5*width, df["Direct_Tax"], width, label="Direct Tax", color="green")
plt.bar(x - 0.5*width, df["GST"], width, label="GST", color="red")
plt.bar(x + 0.5*width, df["Customs"], width, label="Customs", color="blue")
plt.bar(x + 1.5*width, df["Excise"], width, label="Excise", color="yellow")

plt.xticks(x, years, rotation=45)

plt.xlabel("Financial Year")
plt.ylabel("Tax Collection")
plt.title("Tax Collection Trends Across Years")

plt.legend(title="Tax Types")

plt.tight_layout()
plt.show()